# Los troyanos de Júpiter: ¿hasta dónde sirve el análisis lineal?
**Mecánica Clásica UIS · Reto III.1** · Paola Andrea Ríos Campos, Juan Diego Sarmiento Villarreal

Este cuaderno reproduce **todos** los resultados y figuras del informe. Orden:
1. Librería del modelo (`troyanos.py`)
2. Datos (CSV del JPL SBDB)
3. Teoría lineal
4. Barrido en amplitud (PR3C circular plano) y modelo promediado
5. Integración de los 14 809 troyanos reales (Júpiter circular y elíptico, 3D, 2000 años): ~4 min cada uno
6. Verificaciones independientes
7. Figuras y números del informe

## 1. Librería del modelo
Toda la física está en este archivo: ecuaciones, conversiones de elementos orbitales e integradores.

In [ ]:
%%writefile troyanos.py
"""
troyanos.py -- Modelo de los troyanos de Jupiter (Reto III.1, Mecanica Clasica UIS)
Autores: Paola Andrea Rios Campos, Juan Diego Sarmiento Villarreal

Contiene:
  1. Constantes fisicas (JPL) y parametro de masas mu.
  2. Teoria lineal alrededor de L4 (frecuencias de pequenas oscilaciones).
  3. Modelo promediado de la resonancia 1:1 (pendulo no lineal en phi = lambda - lambda_J).
  4. PR3C circular plano en el marco rotante (adimensional) + constante de Jacobi.
  5. Problema restringido 3D en marco inercial con Sol y Jupiter en orbita
     kepleriana (circular o eliptica), integrado para muchos asteroides a la vez (RK4 vectorizado).
  6. Conversiones elementos orbitales <-> vector de estado.

Unidades del modelo inercial: UA y dias (GM en UA^3/dia^2). Resultados se reportan en anos.
"""
import numpy as np

# ---------------------------------------------------------------------------
# 1. Constantes (JPL DE440 / Horizons)
# ---------------------------------------------------------------------------
GM_SUN = 2.9591220828559115e-04      # UA^3/dia^2
GM_JUP = 2.8253458420837780e-07      # UA^3/dia^2 (sistema de Jupiter)
GM_TOT = GM_SUN + GM_JUP
MU = GM_JUP / GM_TOT                 # parametro de masas ~ 9.5388e-4
DAY_PER_YR = 365.25

# Jupiter, elementos osculantes heliocentricos (ecliptica J2000) en la epoca de los datos
# JPL Horizons, 2026-Jun-09 00:00 TDB = JD 2461200.5 (misma epoca que el CSV del SBDB)
EPOCH_JD = 2461200.5
JUP_ELEM = dict(a=5.203080405012691, e=4.822534670267895e-02,
                i=np.radians(1.303446053841260), Om=np.radians(1.005154861704070e+02),
                w=np.radians(2.736746433399965e+02), M=np.radians(1.024675747387010e+02))

N_JUP = np.sqrt(GM_TOT / JUP_ELEM['a']**3)          # movimiento medio [rad/dia]
T_JUP_YR = 2*np.pi/N_JUP/DAY_PER_YR                  # periodo orbital de Jupiter [anos]


# ---------------------------------------------------------------------------
# 2. Teoria lineal alrededor de L4
# ---------------------------------------------------------------------------
def linear_frequencies(mu=MU):
    """Raices de la ecuacion caracteristica lambda^4 + lambda^2 + (27/4) mu (1-mu) = 0.
    Devuelve (omega_lenta, omega_rapida) en unidades de n (movimiento medio)."""
    disc = np.sqrt(1 - 27*mu*(1-mu))
    w_slow = np.sqrt((1 - disc)/2)
    w_fast = np.sqrt((1 + disc)/2)
    return w_slow, w_fast


def linear_periods_yr(mu=MU, TJ=None):
    TJ = T_JUP_YR if TJ is None else TJ
    ws, wf = linear_frequencies(mu)
    return TJ/ws, TJ/wf


# ---------------------------------------------------------------------------
# 3. Modelo promediado (pendulo de la resonancia 1:1)
#    phi'' = -3 mu n^2 sin(phi) [1 - (2-2cos phi)^(-3/2)]
#    Energia: (1/2) phi'^2 + 3 mu n^2 V(phi),  V(phi) = -cos(phi) + 1/(2 sin(phi/2))
# ---------------------------------------------------------------------------
def V_avg(phi):
    return -np.cos(phi) + 1.0/(2.0*np.sin(np.abs(phi)/2.0))


def averaged_period(phi_min, mu=MU, npts=4000):
    """Periodo de libracion (en unidades de 2pi/n, i.e. periodos de Jupiter) de la orbita
    renacuajo cuyo punto de retorno mas cercano a Jupiter es phi_min (rad, 0<phi_min<pi/3).
    Usa la cuadratura T = 2 * int dphi / phidot, con phidot = sqrt(6 mu n^2 (V(phi_min) - V(phi)))."""
    from scipy.optimize import brentq
    from scipy.integrate import quad
    E = V_avg(phi_min)
    # punto de retorno lejano: V(phi_max) = E con phi_max en (pi/3, pi]
    if E >= V_avg(np.pi):           # no es renacuajo: herradura o circulacion
        return np.nan, np.nan
    phi_max = brentq(lambda p: V_avg(p) - E, np.pi/3, np.pi)
    # sustitucion phi = c + h sin(theta) elimina las singularidades de los extremos
    c, h = (phi_max + phi_min)/2, (phi_max - phi_min)/2

    def integrand(th):
        p = c + h*np.sin(th)
        dV = E - V_avg(p)
        return h*np.cos(th)/np.sqrt(max(6*mu*dV, 1e-300))
    I, _ = quad(integrand, -np.pi/2, np.pi/2, limit=400)
    T = 2*I/(2*np.pi)               # n = 1 -> periodo en unidades de T_J
    return T, phi_max


# ---------------------------------------------------------------------------
# 4. PR3C circular plano, marco rotante, adimensional (G(m1+m2)=1, distancia=1, n=1)
# ---------------------------------------------------------------------------
def cr3bp_rhs(t, s, mu=MU):
    x, y, vx, vy = s
    r1 = np.sqrt((x+mu)**2 + y**2)
    r2 = np.sqrt((x-1+mu)**2 + y**2)
    ax = 2*vy + x - (1-mu)*(x+mu)/r1**3 - mu*(x-1+mu)/r2**3
    ay = -2*vx + y - (1-mu)*y/r1**3 - mu*y/r2**3
    return np.array([vx, vy, ax, ay])


def jacobi(s, mu=MU):
    x, y, vx, vy = s
    r1 = np.sqrt((x+mu)**2 + y**2)
    r2 = np.sqrt((x-1+mu)**2 + y**2)
    U = 0.5*(x**2+y**2) + (1-mu)/r1 + mu/r2
    return 2*U - (vx**2+vy**2)


def L4(mu=MU):
    return np.array([0.5-mu, np.sqrt(3)/2])


# ---------------------------------------------------------------------------
# 5. Elementos <-> estado (vectorizado)
# ---------------------------------------------------------------------------
def kepler_E(M, e, tol=1e-13):
    E = np.where(e < 0.8, M, np.pi*np.ones_like(M))
    for _ in range(50):
        f = E - e*np.sin(E) - M
        dE = -f/(1 - e*np.cos(E))
        E = E + dE
        if np.max(np.abs(dE)) < tol:
            break
    return E


def elements_to_state(a, e, i, Om, w, M, gm):
    a, e, i, Om, w, M = map(np.asarray, (a, e, i, Om, w, M))
    E = kepler_E(np.mod(M, 2*np.pi), e)
    cE, sE = np.cos(E), np.sin(E)
    fac = np.sqrt(1 - e**2)
    xp, yp = a*(cE - e), a*fac*sE
    r = a*(1 - e*cE)
    nmot = np.sqrt(gm/a**3)
    vxp, vyp = -a*nmot*sE*a/r, a*nmot*fac*cE*a/r
    cO, sO, cw, sw, ci, si = np.cos(Om), np.sin(Om), np.cos(w), np.sin(w), np.cos(i), np.sin(i)
    R11 = cO*cw - sO*sw*ci; R12 = -cO*sw - sO*cw*ci
    R21 = sO*cw + cO*sw*ci; R22 = -sO*sw + cO*cw*ci
    R31 = sw*si;            R32 = cw*si
    pos = np.stack([R11*xp + R12*yp, R21*xp + R22*yp, R31*xp + R32*yp], axis=-1)
    vel = np.stack([R11*vxp + R12*vyp, R21*vxp + R22*vyp, R31*vxp + R32*vyp], axis=-1)
    return pos, vel


def state_to_elements(r, v, gm):
    """Devuelve a, e, i, Om, varpi, lambda (longitud media). r, v de forma (...,3)."""
    rn = np.linalg.norm(r, axis=-1)
    v2 = np.sum(v*v, axis=-1)
    h = np.cross(r, v)
    hn = np.linalg.norm(h, axis=-1)
    a = 1.0/(2.0/rn - v2/gm)
    evec = np.cross(v, h)/gm - r/rn[..., None]
    e = np.linalg.norm(evec, axis=-1)
    i = np.arccos(np.clip(h[..., 2]/hn, -1, 1))
    Om = np.arctan2(h[..., 0], -h[..., 1])
    # longitud verdadera "dog-leg" y longitud del perihelio
    cO, sO, ci = np.cos(Om), np.sin(Om), np.cos(i)
    # argumento de latitud u: angulo desde el nodo ascendente
    xn = r[..., 0]*cO + r[..., 1]*sO
    yn = (-r[..., 0]*sO + r[..., 1]*cO)*ci + r[..., 2]*np.sin(i)
    u = np.arctan2(yn, xn)
    exn = evec[..., 0]*cO + evec[..., 1]*sO
    eyn = (-evec[..., 0]*sO + evec[..., 1]*cO)*ci + evec[..., 2]*np.sin(i)
    w = np.arctan2(eyn, exn)
    f = u - w
    E = 2*np.arctan2(np.sqrt(1-e)*np.sin(f/2), np.sqrt(1+e)*np.cos(f/2))
    M = E - e*np.sin(E)
    varpi = Om + w
    lam = np.mod(varpi + M, 2*np.pi)
    return a, e, i, Om, varpi, lam


# ---------------------------------------------------------------------------
# 6. Problema restringido 3D con primarios keplerianos (marco inercial baricentrico)
# ---------------------------------------------------------------------------
class Primaries:
    """Sol y Jupiter en orbita kepleriana relativa (two-body exacto).
    eccentric=False -> orbita circular de radio a_J en el plano de la orbita de Jupiter,
    con la misma longitud media en la epoca."""
    def __init__(self, eccentric=True, elem=JUP_ELEM):
        el = dict(elem)
        if not eccentric:
            lam0 = el['Om'] + el['w'] + el['M']
            el['e'] = 0.0
            el['w'] = 0.0
            el['M'] = lam0 - el['Om']
        self.el = el
        self.n = np.sqrt(GM_TOT/el['a']**3)

    def rel(self, t):
        """Posicion y velocidad de Jupiter relativas al Sol en t (dias desde la epoca)."""
        el = self.el
        M = el['M'] + self.n*t
        p, v = elements_to_state(el['a'], el['e'], el['i'], el['Om'], el['w'],
                                 np.atleast_1d(M), GM_TOT)
        return p[0], v[0]

    def positions(self, t):
        d, _ = self.rel(t)
        return -MU*d, (1-MU)*d          # Sol, Jupiter (baricentricos)


def accel(r, t, prim):
    rs, rj = prim.positions(t)
    d1 = r - rs
    d2 = r - rj
    r1 = np.sqrt(np.sum(d1*d1, axis=1))[:, None]
    r2 = np.sqrt(np.sum(d2*d2, axis=1))[:, None]
    return -GM_SUN*d1/r1**3 - GM_JUP*d2/r2**3


def integrate_rk4(r0, v0, prim, t_end, dt, n_out):
    """RK4 de paso fijo para N particulas de prueba a la vez.
    Devuelve tiempos de salida, posiciones y velocidades baricentricas (N_out, N, 3)."""
    nsteps = int(np.ceil(t_end/dt))
    dt = t_end/nsteps
    every = max(1, nsteps//n_out)
    r, v = r0.copy(), v0.copy()
    T, R, Vv = [0.0], [r.copy()], [v.copy()]
    t = 0.0
    for k in range(1, nsteps+1):
        a1 = accel(r, t, prim)
        a2 = accel(r + 0.5*dt*v + dt*dt/8*a1, t + 0.5*dt, prim)
        a3 = accel(r + 0.5*dt*v + dt*dt/8*a2, t + 0.5*dt, prim)
        a4 = accel(r + dt*v + dt*dt/2*a3, t + dt, prim)
        # RK4 para ecuaciones de segundo orden (Runge-Kutta-Nystrom clasico)
        r_new = r + dt*v + dt*dt/6*(a1 + a2 + a3)
        v_new = v + dt/6*(a1 + 2*a2 + 2*a3 + a4)
        r, v = r_new, v_new
        t = k*dt
        if k % every == 0:
            T.append(t); R.append(r.copy()); Vv.append(v.copy())
    return np.array(T), np.array(R), np.array(Vv)


def resonant_angle(T, R, Vv, prim):
    """phi = lambda - lambda_J (heliocentrico, longitudes medias) y a heliocentrico.
    Devuelve phi en (-pi, pi], a (UA), e, i para cada salida y particula."""
    phis, As, Es, Is = [], [], [], []
    for t, r, v in zip(T, R, Vv):
        rs, rj = prim.positions(t)
        d, dv = prim.rel(t)
        vs = -MU*dv
        _, _, _, _, _, lamJ = state_to_elements(d[None, :], dv[None, :], GM_TOT)
        a, e, i, _, _, lam = state_to_elements(r - rs, v - vs, GM_SUN)
        phis.append(np.angle(np.exp(1j*(lam - lamJ[0]))))
        As.append(a); Es.append(e); Is.append(i)
    return np.array(phis), np.array(As), np.array(Es), np.array(Is)


def jacobi_inertial(T, R, Vv, prim):
    """Integral de Jacobi en el marco inercial (solo vale para primarios circulares):
    C_J = v^2/2 - GM_s/r1 - GM_J/r2 - n * (h . k_J), con k_J la normal al plano de Jupiter."""
    d0, dv0 = prim.rel(0.0)
    kJ = np.cross(d0, dv0); kJ /= np.linalg.norm(kJ)
    out = []
    for t, r, v in zip(T, R, Vv):
        rs, rj = prim.positions(t)
        r1 = np.linalg.norm(r - rs, axis=1); r2 = np.linalg.norm(r - rj, axis=1)
        E = 0.5*np.sum(v*v, axis=1) - GM_SUN/r1 - GM_JUP/r2
        h = np.cross(r, v)
        out.append(E - prim.n*(h @ kJ))
    return np.array(out)


def libration_stats(T_days, phi):
    """Amplitud (semi-rango, grados), centro, periodo (anos) y bandera de escape.
    Escape: phi cruza +-180 deg (circula) o cambia de enjambre (cruza 0 por Jupiter)."""
    ph = np.degrees(phi)
    sign0 = np.sign(ph[0])
    escaped = np.any(np.sign(ph) != sign0, axis=0)
    amp = 0.5*(ph.max(axis=0) - ph.min(axis=0))
    cen = 0.5*(ph.max(axis=0) + ph.min(axis=0))
    mean = ph.mean(axis=0)
    # periodo: cruces ascendentes por la media movil (filtrada) -> tiempo medio entre cruces
    tyr = T_days/DAY_PER_YR
    per = np.full(ph.shape[1], np.nan)
    x = ph - mean[None, :]
    up = (x[:-1] < 0) & (x[1:] >= 0)
    for j in range(ph.shape[1]):
        idx = np.where(up[:, j])[0]
        if len(idx) >= 3:
            # interpolacion lineal del cruce
            tc = tyr[idx] + (tyr[idx+1]-tyr[idx])*(-x[idx, j])/(x[idx+1, j]-x[idx, j])
            # descartar cruces espurios por oscilacion rapida: usar mediana de diferencias largas
            dts = np.diff(tc)
            dts = dts[dts > 40]           # el periodo de libracion es >> T_J
            if len(dts) >= 2:
                per[j] = np.median(dts)
    return amp, cen, mean, per, escaped


def smooth_phi(T_days, phi, window_yr=12.0):
    """Promedio movil de ancho ~ T_J para quitar el termino de periodo corto."""
    dt = (T_days[1]-T_days[0])/DAY_PER_YR
    k = max(1, int(round(window_yr/dt)))
    ker = np.ones(k)/k
    # promediar en el plano complejo para no romper el corte en +-pi
    z = np.exp(1j*phi)
    out = np.empty_like(phi)
    for j in range(phi.shape[1]):
        zr = np.convolve(z[:, j].real, ker, mode='same')
        zi = np.convolve(z[:, j].imag, ker, mode='same')
        out[:, j] = np.angle(zr + 1j*zi)
    h = k//2 + 1
    return out[h:-h], T_days[h:-h]


def integrate_track(r0, v0, prim, t_end, dt, out_every_days):
    """Igual que integrate_rk4 pero guarda solo phi, a, e, i (float32) en cada salida:
    permite integrar ~15 000 asteroides sin agotar la memoria."""
    nsteps = int(np.ceil(t_end/dt)); dt = t_end/nsteps
    every = max(1, int(round(out_every_days/dt)))
    r, v = r0.copy(), v0.copy()
    T, PHI, A, E, I = [], [], [], [], []

    def record(t):
        ph, a, e, i = resonant_angle([t], [r], [v], prim)
        T.append(t); PHI.append(ph[0].astype(np.float32)); A.append(a[0].astype(np.float32))
        E.append(e[0].astype(np.float32)); I.append(i[0].astype(np.float32))
    record(0.0)
    t = 0.0
    for k in range(1, nsteps+1):
        a1 = accel(r, t, prim)
        a2 = accel(r + 0.5*dt*v + dt*dt/8*a1, t + 0.5*dt, prim)
        a3 = accel(r + 0.5*dt*v + dt*dt/8*a2, t + 0.5*dt, prim)
        a4 = accel(r + dt*v + dt*dt/2*a3, t + dt, prim)
        r = r + dt*v + dt*dt/6*(a1 + a2 + a3)
        v = v + dt/6*(a1 + 2*a2 + 2*a3 + a4)
        t = k*dt
        if k % every == 0:
            record(t)
    return np.array(T), np.array(PHI), np.array(A), np.array(E), np.array(I)


def load_trojans(csv_path, epoch_cal='2026-06-09.0'):
    """Lee el CSV del JPL SBDB y se queda con los asteroides de la epoca mayoritaria."""
    import pandas as pd
    d = pd.read_csv(csv_path, low_memory=False)
    d = d[d['epoch_cal'] == epoch_cal].reset_index(drop=True)
    r, v = elements_to_state(d.a.values, d.e.values, np.radians(d.i.values),
                             np.radians(d.om.values), np.radians(d.w.values),
                             np.radians(d.ma.values), GM_SUN)
    return d, r, v


def helio_to_bary(r_h, v_h, prim):
    """Heliocentrico -> baricentrico del par Sol-Jupiter: r_bar = r_hel - mu * r_J,hel."""
    dJ, dvJ = prim.rel(0.0)
    return r_h - MU*dJ, v_h - MU*dvJ


In [ ]:
%%writefile run_real.py
"""Integra todos los troyanos reales (epoca 2026-06-09) con primarios circulares o elipticos.
Uso: python run_real.py circular|eliptico [anos] [pasos_por_orbita]"""
import sys, time, numpy as np
from troyanos import *
modelo = sys.argv[1]
anos = float(sys.argv[2]) if len(sys.argv) > 2 else 2000
nst = int(sys.argv[3]) if len(sys.argv) > 3 else 300
csv = sys.argv[4] if len(sys.argv) > 4 else 'datos/sbdb_query_results2.csv'
d, r, v = load_trojans(csv)
prim = Primaries(eccentric=(modelo == 'eliptico'))
rb, vb = helio_to_bary(r, v, prim)
t0 = time.time()
T, PHI, A, E, I = integrate_track(rb, vb, prim, anos*DAY_PER_YR, 2*np.pi/prim.n/nst, DAY_PER_YR)
print(modelo, 'tiempo de computo', time.time()-t0, 's')
np.savez_compressed(f'resultados/real_{modelo}.npz', T=T, PHI=PHI, A=A, E=E, I=I,
                    spkid=d.spkid.values)


## 2. Datos
Sube el archivo `sbdb_query_results2.csv` (JPL Small-Body Database, clase TJN) cuando lo pida la celda. Solo se usan los asteroides con época 2026-06-09 (la mayoría).

In [ ]:
import os
os.makedirs('datos', exist_ok=True); os.makedirs('resultados', exist_ok=True); os.makedirs('figs', exist_ok=True)
if not os.path.exists('datos/sbdb_query_results2.csv'):
    from google.colab import files
    up = files.upload()
    nombre = list(up.keys())[0]
    os.rename(nombre, 'datos/sbdb_query_results2.csv')
import pandas as pd
d = pd.read_csv('datos/sbdb_query_results2.csv', low_memory=False)
print(d.shape)
print(d.epoch_cal.value_counts().head())

## 3. Teoría lineal
Raíces de $\lambda^4+\lambda^2+\tfrac{27}{4}\mu(1-\mu)=0$.

In [ ]:
from troyanos import *
print('mu =', MU)
print('T_J =', T_JUP_YR, 'anos')
ws, wf = linear_frequencies()
print('omega lenta/n =', ws, ' omega rapida/n =', wf)
print('Periodos lineales (libracion, epiciclo) [anos]:', linear_periods_yr())
print('Criterio de Routh: 27 mu (1-mu) =', 27*MU*(1-MU), '< 1 ?')

## 4. Barrido en amplitud
PR3C completo (DOP853) frente al modelo promediado (cuadratura) y la teoría lineal. Tarda ~1 min.

In [ ]:
"""Barrido en amplitud en el PR3C circular plano (marco rotante, adimensional).
Compara: (1) integracion numerica completa, (2) modelo promediado (pendulo), (3) teoria lineal."""
import numpy as np, json
from scipy.integrate import solve_ivp
from troyanos import *

ws, wf = linear_frequencies()
T_lin = 1/ws                               # en periodos de Jupiter
res = []
phis0 = np.radians(np.concatenate([np.arange(59, 30, -1.0), np.arange(30, 23.0, -0.5)]))
for p0 in phis0:
    s0 = np.array([np.cos(p0) - MU, np.sin(p0), 0.0, 0.0])   # en reposo en el marco rotante, r=1 del Sol
    Tav, pmax = averaged_period(p0)
    tend = 2*np.pi*(8*Tav if np.isfinite(Tav) else 400)
    tend = min(max(tend, 2*np.pi*120), 2*np.pi*2000)
    sol = solve_ivp(cr3bp_rhs, (0, tend), s0, method='DOP853', rtol=1e-12, atol=1e-12,
                    dense_output=True)
    t = np.linspace(0, tend, int(tend/(2*np.pi)*40))
    S = sol.sol(t)
    phi = np.arctan2(S[1], S[0] + MU)
    CJ = jacobi(S); dC = np.max(np.abs(CJ - CJ[0]))/abs(CJ[0])
    tdays = t/(2*np.pi)*T_JUP_YR*DAY_PER_YR
    amp, cen, mean, per, esc = libration_stats(tdays, phi[:, None])
    res.append(dict(phi0=np.degrees(p0), amp=float(amp[0]), cen=float(cen[0]), mean=float(mean[0]),
                    phimin=float(np.degrees(phi.min())), phimax=float(np.degrees(phi.max())),
                    T_num=float(per[0]/T_JUP_YR), T_avg=float(Tav),
                    phimax_avg=float(np.degrees(pmax)) if np.isfinite(pmax) else None,
                    esc=bool(esc[0]), dC=float(dC)))
    r = res[-1]
    print(f"{r['phi0']:5.1f} amp={r['amp']:6.2f} Tnum={r['T_num']:.4f} Tavg={r['T_avg']:.4f} "
          f"pmax={r['phimax']:.1f}/{r['phimax_avg']} esc={r['esc']} dC={dC:.1e}", flush=True)
json.dump(dict(T_lin=T_lin, T_JUP_YR=T_JUP_YR, mu=MU, barrido=res), open('resultados/barrido.json', 'w'), indent=1)


## 5. Troyanos reales
Cada corrida integra los 14 809 asteroides a la vez durante 2000 años (~4 min en Colab). Para pruebas rápidas, cambia `ANOS` a 500.

In [ ]:
ANOS = 2000
import subprocess, sys
for modelo in ['circular', 'eliptico']:
    p = subprocess.run([sys.executable, 'run_real.py', modelo, str(ANOS)], capture_output=True, text=True)
    print(p.stdout, p.stderr[-500:])

## 6. Verificaciones independientes
Jacobi, convergencia con el paso, RK4 vs DOP853. Tarda ~5 min.

In [ ]:
"""Verificaciones independientes:
 V1. Ida y vuelta elementos -> estado -> elementos.
 V2. Conservacion de la constante de Jacobi (primarios circulares) en el integrador RK4.
 V3. Convergencia numerica: paso T_J/300 vs T_J/600 (amplitud y periodo de libracion).
 V4. Dos metodos independientes: RK4 vectorizado vs DOP853 adaptativo de SciPy (modelo eliptico).
 V5. Limite mu -> 0 / amplitud -> 0 ya cubierto en run_barrido.py (T_num -> T_lineal)."""
import numpy as np, json
from scipy.integrate import solve_ivp
from troyanos import *
out = {}
d, r, v = load_trojans('datos/sbdb_query_results2.csv')
a, e, i, Om, vp, lam = state_to_elements(r, v, GM_SUN)
lam0 = np.radians(d.om + d.w + d.ma).values
out['V1_err_a'] = float(np.max(abs(a - d.a.values)))
out['V1_err_lambda'] = float(np.max(abs(np.angle(np.exp(1j*(lam - lam0))))))

rng = np.random.default_rng(1)
idx = rng.choice(len(d), 400, replace=False)
anos = 1500
res = {}
for modelo in ['circular', 'eliptico']:
    prim = Primaries(eccentric=(modelo == 'eliptico'))
    rb, vb = helio_to_bary(r[idx], v[idx], prim)
    for nst in [150, 300, 600]:
        T, PHI, A, E, I = integrate_track(rb, vb, prim, anos*DAY_PER_YR, 2*np.pi/prim.n/nst, DAY_PER_YR)
        amp, cen, mean, per, esc = libration_stats(T, PHI.astype(float))
        res[(modelo, nst)] = (amp, per, esc)
    if modelo == 'circular':
        Tt, R, Vv = integrate_rk4(rb, vb, prim, anos*DAY_PER_YR, 2*np.pi/prim.n/300, 300)
        C = jacobi_inertial(Tt, R, Vv, prim)
        dC = np.max(np.abs(C - C[0])/np.abs(C[0]), axis=0)
        out['V2_jacobi_mediana'] = float(np.median(dC)); out['V2_jacobi_max'] = float(np.max(dC))
for modelo in ['circular', 'eliptico']:
    for nst in [150, 300]:
        a1, p1, e1 = res[(modelo, nst)]; a2, p2, e2 = res[(modelo, 600)]
        ok = ~e1 & ~e2 & np.isfinite(p1) & np.isfinite(p2)
        out[f'V3_{modelo}_{nst}_vs_600_dAmp_max_deg'] = float(np.max(abs(a1 - a2)[ok]))
        out[f'V3_{modelo}_{nst}_vs_600_dT_rel_mediana'] = float(np.median(abs(p1 - p2)[ok]/p2[ok]))

# V4: DOP853 en el modelo eliptico para 6 asteroides
prim = Primaries(eccentric=True)
sub = idx[:6]
rb, vb = helio_to_bary(r[sub], v[sub], prim)
def rhs(t, s):
    rr = s[:3][None, :]
    return np.concatenate([s[3:], accel(rr, t, prim)[0]])
tout = np.arange(0, 1500*DAY_PER_YR + 1, DAY_PER_YR)
amps_dop, pers_dop = [], []
for k in range(len(sub)):
    sol = solve_ivp(rhs, (0, tout[-1]), np.concatenate([rb[k], vb[k]]), method='DOP853',
                    rtol=1e-11, atol=1e-13, t_eval=tout)
    R = sol.y[:3].T[:, None, :]; V = sol.y[3:].T[:, None, :]
    ph, *_ = resonant_angle(sol.t, R, V, prim)
    am, ce, me, pe, es = libration_stats(sol.t, ph)
    amps_dop.append(am[0]); pers_dop.append(pe[0])
a_rk, p_rk, _ = res[('eliptico', 300)]
out['V4_amp_DOP853'] = [float(x) for x in amps_dop]
out['V4_amp_RK4'] = [float(x) for x in a_rk[:6]]
out['V4_T_DOP853'] = [float(x) for x in pers_dop]
out['V4_T_RK4'] = [float(x) for x in p_rk[:6]]
json.dump(out, open('resultados/verificacion.json', 'w'), indent=1)
print(json.dumps(out, indent=1))


## 7. Figuras y números del informe
Genera `figs/*.png` y `resultados/resumen.json`.

In [ ]:
"""Genera todas las figuras y la tabla de resultados del informe."""
import numpy as np, json, matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from troyanos import *

C1, C2, C3, C4 = '#2a78d6', '#eb6834', '#1baf7a', '#eda100'
INK, MUTED, GRID = '#222222', '#6b6b6b', '#e4e4e4'
plt.rcParams.update({'font.size': 10, 'axes.edgecolor': MUTED, 'axes.labelcolor': INK,
                     'xtick.color': MUTED, 'ytick.color': MUTED, 'axes.grid': True,
                     'grid.color': GRID, 'grid.linewidth': 0.6, 'axes.spines.top': False,
                     'axes.spines.right': False, 'legend.frameon': False, 'savefig.dpi': 200,
                     'savefig.bbox': 'tight'})
B = json.load(open('resultados/barrido.json'))
Tlin_TJ = B['T_lin']; TLIN = Tlin_TJ*T_JUP_YR
bar = [b for b in B['barrido'] if not b['esc']]
D = np.array([b['amp'] for b in bar]); Tn = np.array([b['T_num'] for b in bar])
Ta = np.array([b['T_avg'] for b in bar])
pmin = np.array([b['phimin'] for b in bar]); pmax = np.array([b['phimax'] for b in bar])
resumen = {'T_lin_yr': TLIN, 'T_fast_yr': linear_periods_yr()[1], 'mu': MU, 'T_J': T_JUP_YR}
# umbrales de validez (interpolacion en el barrido numerico)
for th in [0.01, 0.05, 0.10]:
    resumen[f'D_umbral_{int(th*100)}pct'] = float(np.interp(1+th, Tn/Tlin_TJ, D))
resumen['D_max_renacuajo'] = float(D.max())
resumen['phimin_ultimo_renacuajo'] = float(min(b['phi0'] for b in bar))

# ---- datos reales ----
d, r, v = load_trojans('datos/sbdb_query_results2.csv')
reales = {}
for m in ['circular', 'eliptico']:
    z = np.load(f'resultados/real_{m}.npz'); T = z['T']; P = z['PHI'].astype(float)
    Ps, Ts = smooth_phi(T, P, 12.0)
    amp, cen, mean, per, esc = libration_stats(Ts, Ps)
    esc = esc | libration_stats(T, P)[4]
    reales[m] = dict(amp=amp, cen=cen, per=per, esc=esc, T=T, P=P, inc=np.degrees(z['I'][0]))
R = reales['eliptico']
ok = ~R['esc'] & np.isfinite(R['per'])
amp, per, inc = R['amp'], R['per'], R['inc']
resumen['N_total'] = int(len(amp)); resumen['N_L4'] = int((R['P'][0] > 0).sum())
resumen['N_L5'] = int((R['P'][0] < 0).sum()); resumen['N_escapan'] = int(R['esc'].sum())
resumen['N_escapan_circ'] = int(reales['circular']['esc'].sum())
resumen['N_escapan_ambos'] = int((reales['circular']['esc'] & R['esc']).sum())
resumen['amp_percentiles_5_25_50_75_95'] = np.percentile(amp[~R['esc']], [5, 25, 50, 75, 95]).round(1).tolist()
resumen['per_percentiles_5_25_50_75_95'] = np.percentile(per[ok], [5, 25, 50, 75, 95]).round(1).tolist()
for th in [1, 5, 10]:
    resumen[f'frac_amp_bajo_umbral_{th}pct'] = float(np.mean(amp[~R['esc']] < resumen[f'D_umbral_{th}pct']))
    resumen[f'frac_T_dentro_{th}pct_lineal'] = float(np.mean(np.abs(per[ok]/TLIN - 1) < th/100))
# fase media observada (instantanea, dato puro) vs prediccion lineal 60 deg
ph0 = np.degrees(R['P'][0])
resumen['phi_medio_obs_L4'] = float(ph0[ph0 > 0].mean()); resumen['phi_medio_obs_L5'] = float(-ph0[ph0 < 0].mean())
resumen['phi_medio_obs_L4_err'] = float(ph0[ph0 > 0].std()/np.sqrt((ph0 > 0).sum()))
resumen['phi_medio_obs_L5_err'] = float(ph0[ph0 < 0].std()/np.sqrt((ph0 < 0).sum()))
Pall = np.degrees(R['P'][:, ~R['esc']])
resumen['phi_medio_modelo_L4'] = float(np.mean(Pall[:, Pall[0] > 0]))
resumen['phi_medio_modelo_L5'] = float(-np.mean(Pall[:, Pall[0] < 0]))
# circular vs eliptico
Cc = reales['circular']; both = ok & ~Cc['esc'] & np.isfinite(Cc['per'])
resumen['circ_vs_elip_dAmp_mediana_deg'] = float(np.median(np.abs(Cc['amp'] - amp)[both]))
resumen['circ_vs_elip_dT_mediana_rel'] = float(np.median(np.abs(Cc['per'] - per)[both]/per[both]))
resumen['circ_vs_elip_dAmp_p95_deg'] = float(np.percentile(np.abs(Cc['amp'] - amp)[both], 95))
# inclinacion: periodo mediano a amplitud pequena por rango de inclinacion
resumen['T_amp_lt10_por_inc'] = {}
for lo, hi in [(0, 10), (10, 20), (20, 30), (30, 60)]:
    s = ok & (inc >= lo) & (inc < hi) & (amp < 10)
    resumen['T_amp_lt10_por_inc'][f'{lo}-{hi}'] = [int(s.sum()), float(np.median(per[s]))]
# amplitud predicha por la teoria lineal desde la condicion inicial
a0 = d.a.values; dphi0 = np.radians(np.abs(ph0)) - np.pi/3
Dlin = np.degrees(np.sqrt(dphi0**2 + ((a0 - JUP_ELEM['a'])/(JUP_ELEM['a']*np.sqrt(3*MU)))**2))
resumen['Dlin_vs_Dnum_mediana_rel'] = float(np.median(np.abs(Dlin - amp)[~R['esc']]/amp[~R['esc']]))
json.dump(resumen, open('resultados/resumen.json', 'w'), indent=1)
print(json.dumps(resumen, indent=1))

# ---------------- Figura 1: marco rotante con troyanos reales ----------------
fig, ax = plt.subplots(figsize=(5.2, 5.2))
dJ, _ = Primaries(True).rel(0.0)
thJ = np.arctan2(dJ[1], dJ[0])
x = r[:, 0]*np.cos(thJ) + r[:, 1]*np.sin(thJ); y = -r[:, 0]*np.sin(thJ) + r[:, 1]*np.cos(thJ)
ax.scatter(x, y, s=0.6, color=C1, alpha=0.35, lw=0, label='Troyanos reales (JPL, 2026-06-09)')
th = np.linspace(0, 2*np.pi, 400); aJ = np.linalg.norm(dJ)
ax.plot(aJ*np.cos(th), aJ*np.sin(th), color=MUTED, lw=0.8, ls='--')
ax.plot(0, 0, 'o', color=C4, ms=10, label='Sol'); ax.plot(aJ, 0, 'o', color=C2, ms=7, label='Júpiter')
for nm, ang in [('$L_4$', 60), ('$L_5$', -60), ('$L_3$', 180)]:
    ax.plot(aJ*np.cos(np.radians(ang)), aJ*np.sin(np.radians(ang)), 'x', color=INK, ms=8, mew=2)
    ax.annotate(nm, (aJ*np.cos(np.radians(ang)), aJ*np.sin(np.radians(ang))), xytext=(8, 6),
                textcoords='offset points', fontsize=12, color=INK)
ax.set_aspect('equal'); ax.set_xlabel('x [UA] (marco rotante, Júpiter sobre +x)'); ax.set_ylabel('y [UA]')
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([],[],marker='o',ls='',color=C1,ms=4,label='Troyanos reales (JPL, 2026-06-09)'),Line2D([],[],marker='o',ls='',color=C4,ms=8,label='Sol'),Line2D([],[],marker='o',ls='',color=C2,ms=6,label='Júpiter')],loc='lower left', fontsize=8)
fig.savefig('figs/fig1_marco_rotante.png'); plt.close(fig)

# ---------------- Figura 2: periodo vs amplitud ----------------
fig, ax = plt.subplots(figsize=(6, 3.9))
ax.axhspan(0.95, 1.05, color=GRID, alpha=0.7, lw=0)
ax.plot(D, Tn/Tlin_TJ, 'o-', color=C1, ms=4, lw=1.6, label='PR3C completo (numérico, DOP853)')
ax.plot(D, Ta/Tlin_TJ, '-', color=C2, lw=1.6, label='Modelo promediado (cuadratura)')
ax.axhline(1, color=INK, lw=1.2, ls='--', label=f'Teoría lineal: $T$ = {TLIN:.1f} años')
for th, lab in [(5, '5 %'), (10, '10 %')]:
    Dth = resumen[f'D_umbral_{th}pct']
    ax.axvline(Dth, color=MUTED, lw=0.8, ls=':')
    ax.annotate(f'{lab}: $D$ ≈ {Dth:.0f}°', (Dth, 1.30), xytext=(3, 0), textcoords='offset points',
                fontsize=8, color=MUTED, rotation=90, va='bottom')
ax.set_xlabel('Amplitud de libración $D$ [grados]'); ax.set_ylabel('$T_{\\rm lib}/T_{\\rm lineal}$')
ax.set_xlim(0, 75); ax.set_ylim(0.97, 1.65); ax.legend(fontsize=8, loc='upper left')
fig.savefig('figs/fig2_periodo_amplitud.png'); plt.close(fig)

# ---------------- Figura 3: asimetria del renacuajo ----------------
fig, ax = plt.subplots(figsize=(6, 3.6))
ax.plot(D, pmax, 'o-', color=C1, ms=3.5, lw=1.5, label='$\\varphi_{\\max}$ numérico')
ax.plot(D, pmin, 'o-', color=C2, ms=3.5, lw=1.5, label='$\\varphi_{\\min}$ numérico')
ax.plot(D, 60 + D, '--', color=C1, lw=1.2, label='$60°+D$ (lineal)')
ax.plot(D, 60 - D, '--', color=C2, lw=1.2, label='$60°-D$ (lineal)')
ax.axhline(180, color=MUTED, lw=0.8, ls=':'); ax.annotate('$L_3$', (2, 172), color=MUTED)
ax.set_xlabel('Amplitud de libración $D$ [grados]'); ax.set_ylabel('$\\varphi=\\lambda-\\lambda_J$ [grados]')
ax.set_xlim(0, 75); ax.set_ylim(-15, 190); ax.legend(fontsize=8, ncol=2, loc='lower left')
fig.savefig('figs/fig3_asimetria.png'); plt.close(fig)

# ---------------- Figura 4: troyanos reales T vs D, por inclinacion ----------------
fig, ax = plt.subplots(figsize=(6.2, 4.0))
bins = [(0, 10, '#9ec5f4'), (10, 20, '#5598e7'), (20, 30, '#256abf'), (30, 60, '#0d366b')]
for lo, hi, col in bins:
    s = ok & (inc >= lo) & (inc < hi)
    ax.scatter(amp[s], per[s], s=1.2, color=col, lw=0, alpha=0.6, label=f'$i$ = {lo}–{hi}° ({s.sum()})')
ax.plot(D, Tn*T_JUP_YR, '-', color=C2, lw=2, label='PR3C plano (barrido)')
ax.axhline(TLIN, color=INK, ls='--', lw=1.2, label='Lineal')
ax.set_xlim(0, 60); ax.set_ylim(135, 215)
ax.set_xlabel('Amplitud de libración $D$ [grados]'); ax.set_ylabel('Período de libración [años]')
ax.legend(fontsize=7.5, markerscale=6, loc='upper left', ncol=2)
fig.savefig('figs/fig4_reales_T_vs_D.png'); plt.close(fig)

# ---------------- Figura 5: distribucion de amplitudes reales ----------------
fig, ax = plt.subplots(figsize=(6, 3.4))
hb = np.arange(0, 62, 2)
ax.hist(amp[~R['esc'] & (R['P'][0] > 0)], bins=hb, color=C1, alpha=0.85, label=f"$L_4$ ({resumen['N_L4']})",
        edgecolor='white', lw=0.5)
ax.hist(amp[~R['esc'] & (R['P'][0] < 0)], bins=hb, color=C2, alpha=0.75, label=f"$L_5$ ({resumen['N_L5']})",
        edgecolor='white', lw=0.5)
for th in [5, 10]:
    Dth = resumen[f'D_umbral_{th}pct']
    ax.axvline(Dth, color=INK, lw=1, ls='--')
    ax.annotate(f'error {th} %\n{100*resumen[f"frac_amp_bajo_umbral_{th}pct"]:.1f} % a la izq.', (Dth, ax.get_ylim()[1]*(0.85 if th==5 else 0.6)),
                xytext=(4, 0), textcoords='offset points', fontsize=8, color=INK)
ax.set_xlabel('Amplitud de libración $D$ [grados] (modelo elíptico, 2000 años)'); ax.set_ylabel('Número de asteroides')
ax.legend(fontsize=8, loc='center right')
fig.savefig('figs/fig5_hist_amplitud.png'); plt.close(fig)

# ---------------- Figura 6: ejemplos phi(t) ----------------
fig, ax = plt.subplots(figsize=(6.2, 3.4))
Tyr = R['T']/DAY_PER_YR
s_ok = np.where(ok & (R['P'][0] > 0) & (inc < 10))[0]
j_small = s_ok[np.argmin(np.abs(amp[s_ok] - 5))]; j_big = s_ok[np.argmin(np.abs(amp[s_ok] - 35))]
for j, col in [(j_small, C1), (j_big, C2)]:
    ax.plot(Tyr, np.degrees(R['P'][:, j]), color=col, lw=1.1,
            label=f"{d.full_name.values[j].strip()}  ($D$ = {amp[j]:.0f}°)")
    c = R['cen'][j]
    # solucion lineal con las condiciones iniciales reales: dphi(0) y dphi/dt(0) = -(3/2) n da/a
    w = 2*np.pi/TLIN; nJ = 2*np.pi/T_JUP_YR
    dp0 = np.degrees(R['P'][0, j]) - 60; dpdt0 = np.degrees(-1.5*nJ*(d.a.values[j]-JUP_ELEM['a'])/JUP_ELEM['a'])
    ax.plot(Tyr, 60 + dp0*np.cos(w*Tyr) + dpdt0/w*np.sin(w*Tyr), color=col, lw=1, ls='--', alpha=0.8)
ax.set_xlim(0, 800); ax.set_xlabel('Tiempo [años]'); ax.set_ylabel('$\\varphi$ [grados]')
ax.plot([],[],ls='--',color=MUTED,label='teoría lineal (mismas cond. iniciales)'); ax.set_ylim(25,125); ax.legend(fontsize=8, loc='upper right')
fig.savefig('figs/fig6_ejemplos.png'); plt.close(fig)
json.dump(dict(j_small=int(j_small), j_big=int(j_big), n_small=d.full_name.values[j_small].strip(),
               n_big=d.full_name.values[j_big].strip()), open('resultados/ejemplos.json', 'w'))
print('ok figuras')


In [ ]:
from IPython.display import Image, display
import glob
for f in sorted(glob.glob('figs/*.png')): print(f); display(Image(f, width=600))